# Pertemuan 04 — Hands-on 01: Konvolusi & Pooling from Scratch
**Mata Kuliah:** Deep Learning (IF25-40401) — Program Studi Teknik Informatika, Institut Teknologi Sumatera (ITERA)  
**Materi:** Deep Computer Vision (1): Mekanika Konvolusi, Stride, Padding, dan Operasi Pooling  
**Alokasi Waktu:** ~40 Menit (Tatap Muka Kelas TM 3×50′)  
**Pemetaan Slide:** Bagian 04 (*Slide Frame 33–49*, `materials/pertemuan-04-cnn-from-scratch.tex`)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/informatika-itera/deep-learning-IF25-40401/blob/main/materials/notebooks/01-konvolusi-pooling-from-scratch.ipynb)

---

### Capaian Pembelajaran (Sub-CPMK)
1. **Sub-CPMK 1:** Menjelaskan operasi konvolusi 2D, fungsi kernel/filter, konsep *stride*, *padding* (*valid* vs *same*), dan operasi *pooling* (*max* vs *average*).
2. **Sub-CPMK 2:** Menghitung ukuran *feature map* keluaran ($W_{out}$) dan jumlah parameter konvolusi secara matematis.
3. **Sub-CPMK 3:** Membangun operasi konvolusi dan pooling dari dasar (*from scratch*) dengan NumPy serta memvalidasinya terhadap implementasi PyTorch.
4. **Sub-CPMK 4:** Menganalisis peran *weight sharing* dan *local connectivity* dalam mereduksi parameter serta membentuk invariansi translasi.

---

### Petunjuk Penggunaan
- Jalankan sel secara berurutan dari atas ke bawah (`Shift + Enter`).
- Setiap bagian kunci dilengkapi dengan `assert` numerik yang cocok persis dengan angka perhitungan di slide perkuliahan. Bila seluruh sel lolos tanpa `AssertionError`, implementasi Anda terverifikasi 100% benar.

In [ ]:
# Pastikan torchinfo terinstal (aman diulang pada Colab, Kaggle, maupun lingkungan lokal)
%pip install torchinfo -q

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

# Pengaturan seed untuk reproduktibilitas deterministik
np.random.seed(42)
torch.manual_seed(42)

# Deteksi perangkat komputasi (CUDA -> MPS -> CPU)
device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))

print("=" * 60)
print(f"NumPy Version      : {np.__version__}")
print(f"PyTorch Version    : {torch.__version__}")
print(f"Komputasi Device   : {device}")
print("Lingkungan siap untuk eksperimen konvolusi from scratch!")
print("=" * 60)

## 1. Citra Digital Adalah Matriks Angka

Pada komputer, citra *grayscale* 2D tidak lain adalah matriks berukuran $H \times W$ dengan nilai intensitas piksel (biasanya 0 hingga 255, atau dinormalisasi 0.0 hingga 1.0). 
- Nilai $0$ melambangkan warna **hitam pekat** (tidak ada intensitas cahaya).
- Nilai $1$ (atau $255$) melambangkan warna **putih terang**.

Mari kita buat citra sintetis 16×16 piksel secara deterministik menggunakan NumPy tanpa perlu mengunduh data eksternal.

In [ ]:
# Membuat kanvas citra sintetis 16x16 piksel
citra_sintetis = np.zeros((16, 16), dtype=np.float32)

# Menggambar kotak terang di kuadran kiri atas
citra_sintetis[2:7, 2:7] = 1.0

# Menggambar garis diagonal di kuadran kanan bawah
for k in range(8, 14):
    citra_sintetis[k, k] = 1.0
    if k + 1 < 16:
        citra_sintetis[k, k+1] = 0.7
        citra_sintetis[k+1, k] = 0.7

# Visualisasi citra lengkap dan visualisasi crop 5x5 dengan nilai numerik
fig, axs = plt.subplots(1, 2, figsize=(11, 5))

# Subplot 1: Citra sintetis 16x16
axs[0].imshow(citra_sintetis, cmap='gray', interpolation='nearest')
axs[0].set_title("Citra Sintetis 16×16 (NumPy)")
axs[0].set_xticks(range(0, 16, 2))
axs[0].set_yticks(range(0, 16, 2))
axs[0].grid(color='cyan', linestyle=':', linewidth=0.5)

# Subplot 2: Anotasi nilai piksel pada area 5x5
crop_5x5 = citra_sintetis[2:7, 2:7]
axs[1].imshow(crop_5x5, cmap='gray', interpolation='nearest')
axs[1].set_title("Zoom-in Area 5×5 (Kotak Terang)")
for i in range(5):
    for j in range(5):
        val = crop_5x5[i, j]
        color = "black" if val > 0.5 else "white"
        axs[1].text(j, i, f"{val:.1f}", ha="center", va="center", color=color, fontweight="bold")
axs[1].set_xticks(range(5))
axs[1].set_yticks(range(5))

plt.tight_layout()
plt.show()

## 2. Operasi Konvolusi 2D (Pedagogis *Nested Loop*)

Secara intuitif, operasi konvolusi 2D pada citra bekerja dengan menggeser matriks kecil bernama **kernel** (atau **filter**) ke seluruh bagian citra:
1. Posisikan kernel pada jendela spasial berukuran sama ($K_H \times K_W$).
2. Lakukan perkalian elemen-demi-elemen (*element-wise multiplication*) antara jendela citra dan bobot kernel.
3. Jumlahkan seluruh hasil perkalian tersebut menjadi satu angka skalar. Angka ini mengisi satu sel pada **feature map**.
4. Geser kernel ke posisi berikutnya (*sliding window*) dari kiri ke kanan, lalu atas ke bawah.

Fungsi di bawah sengaja ditulis dengan *nested loop* eksplisit agar alur operasinya transparan dan mudah dipahami mahasiswa.

In [ ]:
def konvolusi_2d(citra: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """
    Melakukan operasi konvolusi 2D sederhana (stride=1, valid padding)
    menggunakan loop eksplisit untuk tujuan pedagogis.
    
    Args:
        citra: Matriks 2D input berukuran (H, W)
        kernel: Matriks 2D kernel/filter berukuran (Kh, Kw)
        
    Returns:
        feature_map: Matriks 2D hasil konvolusi berukuran (H - Kh + 1, W - Kw + 1)
    """
    H, W = citra.shape
    Kh, Kw = kernel.shape
    
    H_out = H - Kh + 1
    W_out = W - Kw + 1
    feature_map = np.zeros((H_out, W_out), dtype=np.float32)
    
    # Geser jendela spasial baris demi baris
    for i in range(H_out):
        for j in range(W_out):
            # 1. Ekstrak jendela spasial dari citra
            jendela = citra[i : i + Kh, j : j + Kw]
            # 2. Perkalian element-wise dan akumulasi penjumlahan
            feature_map[i, j] = np.sum(jendela * kernel)
            
    return feature_map

print("Fungsi konvolusi_2d berhasil didefinisikan.")

## 3. Validasi dengan Contoh Slide Pertemuan 04

Pada slide Pertemuan 04 (*Frame 36–38*, baris 1000–1075), kita menelusuri konvolusi citra $5 \times 5$ dengan kernel diagonal $3 \times 3$:

$$\text{Input} = \begin{pmatrix} 1 & 1 & 1 & 0 & 0 \\ 0 & 1 & 1 & 1 & 0 \\ 0 & 0 & 1 & 1 & 1 \\ 0 & 0 & 1 & 1 & 0 \\ 0 & 1 & 1 & 0 & 0 \end{pmatrix}, \quad \text{Kernel} = \begin{pmatrix} 1 & 0 & 0 \\ 0 & 1 & 0 \\ 0 & 0 & 1 \end{pmatrix}$$

- **Langkah 1 (Posisi 0,0):** $(1\times1) + (1\times1) + (1\times1) = 3$ (*cocok dengan Slide 36*)
- **Langkah 2 (Posisi 0,1):** $(1\times1) + (1\times1) + (1\times1) = 3$ (*cocok dengan Slide 37*)

Mari kita hitung seluruh 9 posisi secara eksak dan lakukan telaah matematis terhadap representasi *feature map*.

In [ ]:
# 1. Input matriks 5x5 dari slide
citra_slide_5x5 = np.array([
    [1, 1, 1, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 1, 1, 1],
    [0, 0, 1, 1, 0],
    [0, 1, 1, 0, 0]
], dtype=np.float32)

# 2. Filter diagonal 3x3 dari slide (mendeteksi garis miring \)
kernel_diagonal = np.array([
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1]
], dtype=np.float32)

# 3. Jalankan konvolusi manual
hasil_fmap = konvolusi_2d(citra_slide_5x5, kernel_diagonal)

print("--- Matriks Input (5x5) ---")
print(citra_slide_5x5.astype(int))
print("\n--- Kernel Diagonal (3x3) ---")
print(kernel_diagonal.astype(int))
print("\n--- Feature Map Eksak (3x3) ---")
print(hasil_fmap.astype(int))

# Verifikasi langkah 1 dan langkah 2 pada slide frame 36 & 37
assert hasil_fmap[0, 0] == 3, "Langkah 1 (0,0) harus bernilai 3 sesuai slide frame 36!"
assert hasil_fmap[0, 1] == 3, "Langkah 2 (0,1) harus bernilai 3 sesuai slide frame 37!"

# Hasil komputasi eksak konvolusi 2D
expected_fmap_exact = np.array([
    [3, 3, 3],
    [1, 3, 2],
    [1, 1, 2]
], dtype=np.float32)

assert np.array_equal(hasil_fmap, expected_fmap_exact), f"Mismatch!\nGot:\n{hasil_fmap}"
print("\n[VALIDASI SUKSES] Perhitungan langkah 1 (nilai 3) dan langkah 2 (nilai 3) lolos verifikasi!")
print("Hasil eksak 9 posisi konvolusi terbukti konsisten dan presisi.")

# Catatan telaah slide:
print("\n[TETAAH PEDAGOGIS SLIDE FRAME 38]")
print("Pada Slide 38 tercantum ilustrasi feature map [[3,3,1],[1,4,3],[1,3,3]].")
print("Perhatikan bahwa kernel diagonal 3x3 biner hanya memiliki 3 angka '1', sehingga pada")
print("citra biner batas nilai maksimum teoritis adalah 3. Angka 4 pada slide adalah ilustrasi")
print("skematis yang nantinya menjadi dasar contoh matriks feature map pada topik pooling (Frame 44).")

## 4. Visualisasi Mekanika *Sliding Window* (9 Posisi)

Untuk memahami bagaimana setiap sel dari *feature map* $3 \times 3$ terbentuk, mari kita visualisasikan seluruh 9 posisi pergeseran kernel di atas citra input $5 \times 5$.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(10, 10))

for r in range(3):
    for c in range(3):
        ax = axes[r, c]
        # Buat visualisasi matriks input
        tampilan = np.zeros((5, 5, 3)) + 0.9  # latar belakang abu muda
        
        # Warnai seluruh input piksel 1
        for i in range(5):
            for j in range(5):
                if citra_slide_5x5[i, j] == 1:
                    tampilan[i, j] = [0.8, 0.85, 1.0]  # biru lembut
                    
        # Warnai jendela kernel aktif
        for ki in range(3):
            for kj in range(3):
                pi, pj = r + ki, c + kj
                if kernel_diagonal[ki, kj] == 1 and citra_slide_5x5[pi, pj] == 1:
                    tampilan[pi, pj] = [0.4, 0.9, 0.4]  # hijau (match!)
                elif kernel_diagonal[ki, kj] == 1:
                    tampilan[pi, pj] = [1.0, 0.8, 0.8]  # merah muda (kernel 1 tapi piksel 0)
                else:
                    tampilan[pi, pj] = [1.0, 1.0, 0.6]  # kuning (kernel 0)

        ax.imshow(tampilan, interpolation='nearest')
        
        # Gambar garis batas jendela kernel
        rect = plt.Rectangle((c - 0.5, r - 0.5), 3, 3, fill=False, edgecolor='crimson', linewidth=2.5)
        ax.add_patch(rect)
        
        # Tampilkan teks nilai piksel
        for i in range(5):
            for j in range(5):
                ax.text(j, i, str(int(citra_slide_5x5[i, j])), ha='center', va='center',
                        fontsize=9, color='black', fontweight='bold')
                
        skalar_hasil = int(hasil_fmap[r, c])
        ax.set_title(f"Pos ({r},{c}) -> Hasil = {skalar_hasil}", fontsize=10, fontweight='bold')
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle("Visualisasi 9 Langkah Sliding Window: Kernel 3×3 pada Citra 5×5", fontsize=14, y=0.99, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Verifikasi Menggunakan PyTorch `torch.nn.functional.conv2d`

Di industri dan riset, kita tidak menggunakan loop Python manual karena lambat. PyTorch menyediakan operasi C++/CUDA teroptimasi tinggi via `F.conv2d`.

> **Catatan Margin Penting (Konvolusi vs Cross-Correlation):**  
> Dalam matematika murni, operasi konvolusi mensyaratkan kernel **dibalik 180°** (*flipped*) sebelum perkalian dot:
> $$(f * g)(t) = \int_{-\infty}^{\infty} f(\tau) g(t - \tau) d\tau$$
> Namun dalam Deep Learning, bobot kernel dipelajari secara otomatis melalui *backpropagation*. Membalik atau tidak membalik kernel menghasilkan representasi ekuivalen karena gradien akan otomatis menyesuaikan orientasi bobot.  
> Oleh karena itu, seluruh pustaka Deep Learning modern (PyTorch, TensorFlow, JAX) secara teknis mengimplementasikan **Cross-Correlation** (tanpa membalik kernel), meskipun secara konvensi tetap disebut **Konvolusi**.

In [ ]:
# PyTorch F.conv2d membutuhkan format tensor 4D: (batch_size, channels, height, width)
tensor_citra = torch.tensor(citra_slide_5x5, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
tensor_kernel = torch.tensor(kernel_diagonal, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

# Jalankan konvolusi PyTorch
torch_out = F.conv2d(tensor_citra, tensor_kernel, stride=1, padding=0)
hasil_torch_np = torch_out.squeeze().numpy()

print(f"Bentuk tensor input  : {tensor_citra.shape}")
print(f"Bentuk tensor kernel : {tensor_kernel.shape}")
print(f"Bentuk tensor output : {torch_out.shape}")
print("\nFeature map PyTorch F.conv2d:")
print(hasil_torch_np.astype(int))

# Validasi assert kesamaan implementasi NumPy vs PyTorch
assert np.allclose(hasil_fmap, hasil_torch_np), "Hasil NumPy berbeda dengan PyTorch!"
print("\n[VALIDASI SUKSES] Hasil konvolusi manual NumPy identik dengan PyTorch F.conv2d!")

## 6. Parameter Konvolusi (1): Stride ($S$)

*Stride* adalah besaran langkah pergeseran kernel pada setiap iterasi.
- $S = 1$: Kernel bergeser 1 piksel per langkah (resolusi keluaran maksimal).
- $S > 1$: Kernel melompati piksel, menghasilkan *downsampling* spasial secara langsung tanpa memerlukan pooling layer terpisah.

Mari kita perluas fungsi konvolusi kita untuk mendukung *stride*.

In [ ]:
def konvolusi_2d_stride(citra: np.ndarray, kernel: np.ndarray, stride: int = 1) -> np.ndarray:
    """Konvolusi 2D dengan parameter stride."""
    H, W = citra.shape
    Kh, Kw = kernel.shape
    
    H_out = (H - Kh) // stride + 1
    W_out = (W - Kw) // stride + 1
    
    output = np.zeros((H_out, W_out), dtype=np.float32)
    for i in range(H_out):
        for j in range(W_out):
            r_start = i * stride
            c_start = j * stride
            jendela = citra[r_start : r_start + Kh, c_start : c_start + Kw]
            output[i, j] = np.sum(jendela * kernel)
            
    return output

# Uji pada citra 7x7 dengan kernel 3x3 untuk stride 1, 2, 3
citra_7x7 = np.ones((7, 7), dtype=np.float32)
kernel_3x3 = np.ones((3, 3), dtype=np.float32)

print("Eksperimen Pengaruh Stride pada Citra 7x7 dan Kernel 3x3:")
print("-" * 55)
for s in [1, 2, 3]:
    out = konvolusi_2d_stride(citra_7x7, kernel_3x3, stride=s)
    # Verifikasi dengan PyTorch
    t_in = torch.tensor(citra_7x7).unsqueeze(0).unsqueeze(0)
    t_k = torch.tensor(kernel_3x3).unsqueeze(0).unsqueeze(0)
    t_out = F.conv2d(t_in, t_k, stride=s).squeeze().numpy()
    
    assert np.array_equal(out, t_out), f"Stride {s} gagal!"
    print(f"Stride S = {s} -> Ukuran Output = {out.shape[0]}×{out.shape[1]} | PyTorch Match: OK")

## 7. Parameter Konvolusi (2): Padding ($P$)

Setiap kali konvolusi diterapkan tanpa padding (*valid padding*, $P=0$), dimensi spasial akan menyusut sebesar $K - 1$. Pada jaringan dalam, citra akan cepat menyusut menjadi $1 \times 1$ dan informasi di tepian citra (*border*) terbuang karena jarang dilintasi kernel.

Untuk mengatasi ini, kita menambahkan bantalan angka nol di sekeliling citra (**zero padding**).
- **Valid Padding ($P=0$):** Tanpa padding, ukuran output mengecil: $W_{out} = W - K + 1$.
- **Same Padding:** Menjaga dimensi spasial output sama persis dengan input ($W_{out} = W$).
  Untuk kernel ganjil $K$, padding yang dibutuhkan:
  $$P = \frac{K - 1}{2}$$
  - Kernel $3 \times 3 \implies P = 1$
  - Kernel $5 \times 5 \implies P = 2$

In [ ]:
def konvolusi_2d_lengkap(citra: np.ndarray, kernel: np.ndarray, stride: int = 1, padding: int = 0) -> np.ndarray:
    """Konvolusi 2D lengkap dengan dukungan padding dan stride."""
    if padding > 0:
        citra_berbantalan = np.pad(citra, pad_width=padding, mode='constant', constant_values=0)
    else:
        citra_berbantalan = citra
        
    return konvolusi_2d_stride(citra_berbantalan, kernel, stride=stride)

# Uji 3 skenario slide (Frame 40 / baris 1166-1175):
# 1. LeNet C1 (Valid): W=32, K=5, S=1, P=0 -> W_out = 28
# 2. Same Conv       : W=32, K=3, S=1, P=1 -> W_out = 32
# 3. Stride 2        : W=32, K=3, S=2, P=1 -> W_out = 16

dummy_32x32 = np.ones((32, 32), dtype=np.float32)
k3 = np.ones((3, 3), dtype=np.float32)
k5 = np.ones((5, 5), dtype=np.float32)

out_c1 = konvolusi_2d_lengkap(dummy_32x32, k5, stride=1, padding=0)
out_same = konvolusi_2d_lengkap(dummy_32x32, k3, stride=1, padding=1)
out_s2 = konvolusi_2d_lengkap(dummy_32x32, k3, stride=2, padding=1)

print("Verifikasi Dimensi Output Sesuai Slide Frame 40:")
print(f"1. Valid Conv (32x32, K=5, S=1, P=0) -> Output: {out_c1.shape}  [Target: 28x28]")
print(f"2. Same Conv  (32x32, K=3, S=1, P=1) -> Output: {out_same.shape}[Target: 32x32]")
print(f"3. Stride 2   (32x32, K=3, S=2, P=1) -> Output: {out_s2.shape}  [Target: 16x16]")

assert out_c1.shape == (28, 28), "Ukuran C1 salah!"
assert out_same.shape == (32, 32), "Ukuran Same Conv salah!"
assert out_s2.shape == (16, 16), "Ukuran Stride 2 salah!"
print("[VALIDASI SUKSES] Seluruh dimensi spasial terbukti presisi!")

## 8. Formula Universal Ukuran Output ($W_{out}$)

Rumus universal dimensi spasial konvolusi 2D:
$$W_{out} = \left\lfloor \frac{W_{in} - K + 2P}{S} \right\rfloor + 1$$

Mari kita buat fungsi pembantu matematis dan validasi terhadap `nn.Conv2d` untuk 6 konfigurasi arsitektur CNN populer.

In [ ]:
def hitung_ukuran_output(W_in: int, K: int, P: int = 0, S: int = 1) -> int:
    """Menghitung ukuran spasial output konvolusi 2D sesuai formula slide."""
    return int(np.floor((W_in - K + 2 * P) / S)) + 1

# 6 Kombinasi uji: LeNet, VGG, ResNet stem, AlexNet
kombinasi_uji = [
    # (Nama, W_in, K, P, S, target)
    ("LeNet C1 (MNIST)", 32, 5, 0, 1, 28),
    ("LeNet C3", 14, 5, 0, 1, 10),
    ("VGG Blok 1", 224, 3, 1, 1, 224),
    ("ResNet-18 Stem", 224, 7, 3, 2, 112),
    ("AlexNet Conv1", 227, 11, 0, 4, 55),
    ("Custom Downsample", 32, 3, 1, 2, 16)
]

print("Tabel Verifikasi Formula Universal Output Shape:")
print(f"{'Nama Layer':<20} | {'W_in':<5} | {'K':<3} | {'P':<3} | {'S':<3} | {'Rumus':<6} | {'nn.Conv2d':<10} | {'Status'}")
print("-" * 75)

for nama, w, k, p, s, target in kombinasi_uji:
    hasil_rumus = hitung_ukuran_output(w, k, p, s)
    
    # Validasi nyata terhadap PyTorch nn.Conv2d
    layer_conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=k, stride=s, padding=p)
    tensor_uji = torch.zeros(1, 1, w, w)
    hasil_pytorch = layer_conv(tensor_uji).shape[-1]
    
    assert hasil_rumus == target == hasil_pytorch, f"Gagal pada {nama}!"
    print(f"{nama:<20} | {w:<5} | {k:<3} | {p:<3} | {s:<3} | {hasil_rumus:<6} | {hasil_pytorch:<10} | PASS")

print("\n[VALIDASI SUKSES] Seluruh 6 konfigurasi lolos validasi matematis dan PyTorch!")

## 9. Formula Parameter Layer Konvolusi

Slide Pertemuan 04 (*Frame 41*, baris 1182–1205) merumuskan total parameter layer konvolusi:
$$\text{Weights} = K_H \times K_W \times C_{in} \times C_{out}, \qquad \text{Biases} = C_{out}$$
$$\mathbf{\text{Total Parameter}} = (K_H \times K_W \times C_{in} + 1) \times C_{out}$$

> **Prinsip Fundamental:** Ukuran spasial citra ($H \times W$) **sama sekali tidak mempengaruhi** jumlah parameter konvolusi! Ini adalah pilar utama efisiensi parameter (*weight sharing*).

In [ ]:
def hitung_parameter_conv2d(K: int, Cin: int, Cout: int, bias: bool = True) -> int:
    """Menghitung jumlah parameter bobot dan bias pada layer Conv2D."""
    b = 1 if bias else 0
    return (K * K * Cin + b) * Cout

# Uji terhadap 2 angka kunci pada slide frame 41:
# Contoh 1: Input RGB (Cin=3), Cout=16, K=3 -> (3x3x3 + 1) x 16 = 448
param_rgb = hitung_parameter_conv2d(K=3, Cin=3, Cout=16, bias=True)
pt_conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, bias=True)
pt_param1 = sum(p.numel() for p in pt_conv1.parameters())

# Contoh 2: Input 64 kanal (Cin=64), Cout=128, K=3 -> (3x3x64 + 1) x 128 = 73.856
param_deep = hitung_parameter_conv2d(K=3, Cin=64, Cout=128, bias=True)
pt_conv2 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, bias=True)
pt_param2 = sum(p.numel() for p in pt_conv2.parameters())

print("Verifikasi Jumlah Parameter Conv2D Sesuai Slide:")
print(f"1. Conv2D (3x3, Cin=3,  Cout=16)  -> Rumus: {param_rgb:<6} | PyTorch: {pt_param1:<6} [Target: 448]")
print(f"2. Conv2D (3x3, Cin=64, Cout=128) -> Rumus: {param_deep:<6} | PyTorch: {pt_param2:<6} [Target: 73.856]")

assert param_rgb == 448 == pt_param1, "Parameter contoh 1 tidak cocok!"
assert param_deep == 73856 == pt_param2, "Parameter contoh 2 tidak cocok!"
print("\n[VALIDASI SUKSES] Formula parameter terbukti akurat 100%!")

## 10. Operasi Pooling: Max Pooling vs Average Pooling

Pooling berfungsi mereduksi ukuran spasial (*downsampling*) sambil mempertahankan informasi penting.
- **Max Pooling:** Mengambil nilai maksimum pada setiap blok jendela ($2 \times 2$). Mengutamakan aktivasi fitur yang paling menonjol (*salient features*).
- **Average Pooling:** Mengambil nilai rata-rata pada setiap blok. Meratakan informasi latar belakang.

Mari kita replikasi contoh numerik dari slide Pertemuan 04 (*Frame 44–46*, baris 1284–1361) pada matriks $4 \times 4$:
$$\begin{pmatrix} 3 & 3 & 2 & 1 \\ 1 & 4 & 3 & 0 \\ 1 & 2 & 3 & 3 \\ 0 & 1 & 1 & 0 \end{pmatrix}$$

In [ ]:
# Matriks 4x4 persis dari slide
fmap_slide_4x4 = np.array([
    [3, 3, 2, 1],
    [1, 4, 3, 0],
    [1, 2, 3, 3],
    [0, 1, 1, 0]
], dtype=np.float32)

def max_pooling_2d(citra: np.ndarray, pool_size: int = 2, stride: int = 2) -> np.ndarray:
    H, W = citra.shape
    H_out = (H - pool_size) // stride + 1
    W_out = (W - pool_size) // stride + 1
    out = np.zeros((H_out, W_out), dtype=np.float32)
    for i in range(H_out):
        for j in range(W_out):
            r, c = i * stride, j * stride
            out[i, j] = np.max(citra[r : r + pool_size, c : c + pool_size])
    return out

def avg_pooling_2d(citra: np.ndarray, pool_size: int = 2, stride: int = 2) -> np.ndarray:
    H, W = citra.shape
    H_out = (H - pool_size) // stride + 1
    W_out = (W - pool_size) // stride + 1
    out = np.zeros((H_out, W_out), dtype=np.float32)
    for i in range(H_out):
        for j in range(W_out):
            r, c = i * stride, j * stride
            out[i, j] = np.mean(citra[r : r + pool_size, c : c + pool_size])
    return out

hasil_max = max_pooling_2d(fmap_slide_4x4, 2, 2)
hasil_avg = avg_pooling_2d(fmap_slide_4x4, 2, 2)

# Target dari slide frame 44 dan 45
target_max = np.array([[4.0, 3.0], [2.0, 3.0]], dtype=np.float32)
target_avg = np.array([[2.75, 1.5], [1.0, 1.75]], dtype=np.float32)

print("Hasil Max Pooling Manual:")
print(hasil_max)
print("\nHasil Average Pooling Manual:")
print(hasil_avg)

# Validasi assert NumPy manual
assert np.allclose(hasil_max, target_max), "Max pooling manual salah!"
assert np.allclose(hasil_avg, target_avg), "Avg pooling manual salah!"

# Validasi dengan PyTorch F.max_pool2d dan F.avg_pool2d
t_pool_in = torch.tensor(fmap_slide_4x4).unsqueeze(0).unsqueeze(0)
pt_max = F.max_pool2d(t_pool_in, kernel_size=2, stride=2).squeeze().numpy()
pt_avg = F.avg_pool2d(t_pool_in, kernel_size=2, stride=2).squeeze().numpy()

assert np.allclose(hasil_max, pt_max), "PyTorch max pool mismatch!"
assert np.allclose(hasil_avg, pt_avg), "PyTorch avg pool mismatch!"

print("\n[VALIDASI SUKSES] Hasil Max Pool [[4,3],[2,3]] dan Avg Pool [[2.75,1.5],[1,1.75]] cocok 100% dengan slide!")

## 11. Karakteristik Pooling: Invariansi Translasi Lokal

Salah satu keunggulan terbesar Max Pooling adalah memberikan **ketahanan terhadap pergeseran posisi kecil** (*local translation invariance*). Bila sebuah fitur bergeser 1 piksel di dalam jendela pooling, nilai aktivasi maksimum yang lolos tetap sama!

$$\max(0, 8, 1, 0) = \max(8, 0, 0, 1) = 8$$

In [ ]:
# Demonstrasi sinyal 1D
sinyal_a = np.array([0, 8, 1, 0])
sinyal_b = np.array([8, 0, 0, 1])  # Bergeser posisi tapi masih dalam jendela 4 elemen

print(f"Max sinyal A: {np.max(sinyal_a)} | Max sinyal B: {np.max(sinyal_b)}")
assert np.max(sinyal_a) == np.max(sinyal_b) == 8, "Invariansi gagal!"

# Demonstrasi visual pada pola 2D kecil
pola_asli = np.array([
    [0, 0, 0, 0],
    [0, 9, 0, 0],
    [0, 0, 0, 0],
    [0, 0, 0, 0]
], dtype=np.float32)

pola_geser = np.array([
    [0, 0, 0, 0],
    [0, 0, 9, 0],  # Titik terang bergeser 1 piksel ke kanan
    [0, 0, 0, 0],
    [0, 0, 0, 0]
], dtype=np.float32)

pool_asli = max_pooling_2d(pola_asli, 2, 2)
pool_geser = max_pooling_2d(pola_geser, 2, 2)

fig, axs = plt.subplots(2, 2, figsize=(7, 6))
axs[0, 0].imshow(pola_asli, cmap='viridis', vmin=0, vmax=9)
axs[0, 0].set_title("Pola Asli (Piksel di (1,1))")
axs[0, 1].imshow(pola_geser, cmap='viridis', vmin=0, vmax=9)
axs[0, 1].set_title("Pola Bergeser (Piksel di (1,2))")

axs[1, 0].imshow(pool_asli, cmap='viridis', vmin=0, vmax=9)
axs[1, 0].set_title("Max Pool Asli -> [[9, 0], [0, 0]]")
axs[1, 1].imshow(pool_geser, cmap='viridis', vmin=0, vmax=9)
axs[1, 1].set_title("Max Pool Geser -> [[0, 9], [0, 0]]")

for ax in axs.ravel():
    ax.set_xticks(range(ax.get_images()[0].get_array().shape[1]))
    ax.set_yticks(range(ax.get_images()[0].get_array().shape[0]))

plt.tight_layout()
plt.show()

## 12. Eksperimen Bank Filter Klasik pada Citra Nyata

Sebelum era Deep Learning, para periset Computer Vision merancang bobot kernel secara manual (*handcrafted features*) untuk mengekstrak tepi, tekstur, dan sudut. Pada CNN, bobot-bobot filter semacam ini **dipelajari secara otomatis** dari data!

Mari kita terapkan 5 filter klasik pada citra `grace_hopper.jpg` (tersedia *built-in* pada library Matplotlib tanpa unduhan internet):
1. **Sobel-X:** Deteksi tepi vertikal ($[-1, 0, 1]$)
2. **Sobel-Y:** Deteksi tepi horizontal ($[-1, -2, -1]^T$)
3. **Sharpen:** Penajaman kontur
4. **Gaussian/Box Blur:** Penghalusan citra
5. **Diagonal Edge:** Deteksi tepi miring

In [ ]:
import matplotlib.cbook as cbook

# Muat citra bawaan matplotlib
img_raw = plt.imread(cbook.get_sample_data('grace_hopper.jpg'))

# Konversi citra RGB ke grayscale (Luminance: 0.299 R + 0.587 G + 0.114 B)
img_gray = (0.2989 * img_raw[:, :, 0] + 0.5870 * img_raw[:, :, 1] + 0.1140 * img_raw[:, :, 2]) / 255.0

# Ambil crop 200x200 di area wajah agar komputasi cepat dan detail terlihat
crop_hopper = img_gray[120:320, 180:380].astype(np.float32)

# Definisi bank filter 3x3 klasik
bank_filter = {
    "Original": None,
    "Sobel-X (Tepi Vertikal)": np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32),
    "Sobel-Y (Tepi Horizontal)": np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float32),
    "Sharpen (Penajaman)": np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=np.float32),
    "Box Blur (Penghalusan)": np.ones((3, 3), dtype=np.float32) / 9.0,
    "Laplacian / Diagonal": np.array([[2, -1, -1], [-1, 2, -1], [-1, -1, 2]], dtype=np.float32)
}

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

for idx, (nama, kernel) in enumerate(bank_filter.items()):
    ax = axes[idx]
    if kernel is None:
        ax.imshow(crop_hopper, cmap='gray')
    else:
        # Terapkan konvolusi lengkap (padding=1 untuk same-size)
        fmap = konvolusi_2d_lengkap(crop_hopper, kernel, stride=1, padding=1)
        ax.imshow(fmap, cmap='gray')
    ax.set_title(nama, fontsize=11, fontweight='bold')
    ax.axis('off')

plt.suptitle("Penerapan Bank Filter Konvolusi pada Wajah Grace Hopper (Crop 200×200)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 13. Ringkasan Konsep & Tugas Terstruktur (TT 3×60′)

### Ringkasan Capaian Pembelajaran:
1. **Konvolusi 2D** adalah perkalian *element-wise* antara kernel spasial dan jendela citra yang digeser ke seluruh area input.
2. **Dimensi Spasial Output:** $W_{out} = \lfloor \frac{W_{in} - K + 2P}{S} \rfloor + 1$.
3. **Total Parameter Layer Konvolusi:** $(K \times K \times C_{in} + 1) \times C_{out}$. Jumlah parameter tidak bergantung pada ukuran citra ($H \times W$).
4. **Max Pooling** mereduksi dimensi sekaligus memberikan ketahanan (*invariance*) terhadap pergeseran posisi kecil.
5. **Cross-Correlation:** Yang diimplementasikan oleh PyTorch dan framework deep learning modern secara teknis adalah operasi korelasi silang tanpa pembalikan kernel.

---

### Soal Latihan Mandiri (Tugas Terstruktur):
1. **Latihan 1 — Dilated Convolution:**  
   Buat fungsi `konvolusi_dilated_2d(citra, kernel, dilation=2)` dari nol dengan NumPy. Bandingkan receptive field efektif kernel $3 \times 3$ dengan dilation=1 vs dilation=2!
2. **Latihan 2 — Aritmetika Arsitektur VGG-16:**  
   Hitung output shape dan jumlah parameter untuk Blok 1 arsitektur VGG-16 yang menerima input RGB $224 \times 224$:
   - Conv 1: 64 filter $3 \times 3$, padding 1, stride 1
   - Conv 2: 64 filter $3 \times 3$, padding 1, stride 1
   - MaxPool: kernel $2 \times 2$, stride 2
3. **Latihan 3 — Global Average Pooling (GAP):**  
   Implementasikan fungsi `global_average_pooling_2d(feature_maps)` yang menerima tensor feature map berukuran $(C, H, W)$ dan menghasilkan vektor fitur representasi $(C,)$ tanpa bobot latih!